In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import pickle


In [8]:
def load_eeg_parquet(path, downsample_factor=10):
    """Load raw EEG parquet, select POW columns, and aggregate to trial-level.

    Args:
        path: Path to the eeg.parquet file
        downsample_factor: Take every Nth row before aggregation to reduce memory

    Returns:
        DataFrame with columns: pid, movement, block_id, nback + 70 POW features
    """
    df = pd.read_parquet(path)

    # Select POW columns (features) and metadata
    pow_cols = [c for c in df.columns if c.startswith("POW.")]
    meta_cols = ["subject", "test", "phase"]
    df = df[pow_cols + meta_cols]

    # Drop rows with any NaN in POW columns
    df = df.dropna(subset=pow_cols)

    # Downsample: take every Nth row within each group
    df["__row_idx"] = df.groupby(meta_cols).cumcount()
    df = df[df["__row_idx"] % downsample_factor == 0]
    df = df.drop(columns=["__row_idx"])

    # Aggregate (mean) per trial
    agg_df = df.groupby(meta_cols)[pow_cols].mean().reset_index()

    # Rename to match expected schema
    agg_df = agg_df.rename(columns={
        "subject": "pid",
        "phase": "block_id",
        "test": "nback",
    })

    # Add dummy 'movement' column to match expected metadata format
    agg_df["movement"] = 0

    # Reorder columns: metadata first, then features
    feature_cols = [c for c in agg_df.columns if c not in ["pid", "movement", "block_id", "nback"]]
    agg_df = agg_df[["pid", "movement", "block_id", "nback"] + feature_cols]

    print(f"[EEG parquet] Loaded {len(agg_df)} trials from {path}")
    print(f"[EEG parquet] Features per trial: {len(feature_cols)}")
    print(f"[EEG parquet] Subjects: {agg_df['pid'].nunique()}")
    print(f"[EEG parquet] N-back levels: {sorted(agg_df['nback'].unique())}")

    return agg_df

In [9]:
def train_pipeline(data, label):
    """Fits Scaler -> PCA -> RandomForest on one dataset and returns the
    fitted objects plus a short report."""
    if isinstance(data, str):
        df = pd.read_csv(data)
    else:
        df = data

    X = df.drop(columns=METADATA_COLS)
    y = df[TARGET_COL]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=0.95)
    X_pca = pca.fit_transform(X_scaled)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_pca, y)

    print(f"[{label}] Original features: {X.shape[1]}")
    print(f"[{label}] PCA features: {X_pca.shape[1]}")
    print(f"[{label}] Explained variance: {pca.explained_variance_ratio_.sum():.4f}")
    print(f"[{label}] Model expects: {model.n_features_in_}")

    return {"scaler": scaler, "pca": pca, "model": model, "classes": model.classes_}

In [10]:
# ── Ensemble / agreement filter ──────────────────────────────────────────────

def ensemble_predict(X_fnirs_raw, X_eeg_raw, pipeline_fnirs, pipeline_eeg, eps=1e-12):
    """
    Combine predictions from the fNIRS and EEG pipelines using a
    maximum-likelihood (product-of-experts) combination rule.

    X_fnirs_raw, X_eeg_raw : raw (unscaled) feature matrices for the SAME
                              samples/trials, in the same row order, one per
                              modality's feature space.

    Why multiply instead of average:
      Each model outputs P_fnirs(y|x_fnirs) and P_eeg(y|x_eeg) — independent
      estimates of the same latent label y from two different modalities. If
      we treat x_fnirs and x_eeg as conditionally independent given y (the
      standard assumption behind combining independent classifiers), then by
      Bayes' rule the joint likelihood of y given both modalities is
      proportional to the PRODUCT of the two per-model likelihoods:

          P(y | x_fnirs, x_eeg)  ∝  P_fnirs(y | x_fnirs) * P_eeg(y | x_eeg)

      Renormalizing that product over y gives the maximum-likelihood combined
      posterior. This is a sharper combination than averaging: if either
      model assigns near-zero probability to a class, the product drives the
      combined probability toward zero for that class too, so confident
      disagreement is punished rather than diluted. Agreement between two
      confident models compounds into a high combined probability
      (mirrors "both models said x, so x is extremely likely").

    Returns a DataFrame with one row per sample:
      pred_fnirs, pred_eeg, agree, final_pred, confidence (MLE posterior prob)
    """
    # Transform each modality through its own scaler + PCA
    X_fnirs_pca = pipeline_fnirs["pca"].transform(pipeline_fnirs["scaler"].transform(X_fnirs_raw))
    X_eeg_pca = pipeline_eeg["pca"].transform(pipeline_eeg["scaler"].transform(X_eeg_raw))

    proba_fnirs = pipeline_fnirs["model"].predict_proba(X_fnirs_pca)
    proba_eeg = pipeline_eeg["model"].predict_proba(X_eeg_pca)

    # Align class ordering between the two models (they might differ if
    # one modality's training data is missing a class)
    classes = sorted(set(pipeline_fnirs["classes"]) | set(pipeline_eeg["classes"]))
    class_index = {c: i for i, c in enumerate(classes)}

    def expand(proba, model_classes):
        # Classes a model never saw get eps probability rather than 0, so
        # they don't permanently zero out that column in the product step.
        out = np.full((proba.shape[0], len(classes)), eps)
        for i, c in enumerate(model_classes):
            out[:, class_index[c]] = proba[:, i]
        return out

    proba_fnirs_full = expand(proba_fnirs, pipeline_fnirs["classes"])
    proba_eeg_full = expand(proba_eeg, pipeline_eeg["classes"])

    pred_fnirs = np.array(classes)[proba_fnirs_full.argmax(axis=1)]
    pred_eeg = np.array(classes)[proba_eeg_full.argmax(axis=1)]
    agree = pred_fnirs == pred_eeg

    # Maximum-likelihood combination: multiply the two likelihood vectors
    # (equivalently, sum their logs — done here in log-space for stability)
    log_combined = np.log(proba_fnirs_full + eps) + np.log(proba_eeg_full + eps)
    # Renormalize (softmax over the summed log-likelihoods) to get a proper
    # posterior distribution over classes
    log_combined -= log_combined.max(axis=1, keepdims=True)  # stability
    combined = np.exp(log_combined)
    combined /= combined.sum(axis=1, keepdims=True)

    final_pred = np.array(classes)[combined.argmax(axis=1)]
    confidence = combined.max(axis=1)

    return pd.DataFrame({
        "pred_fnirs": pred_fnirs,
        "pred_eeg": pred_eeg,
        "agree": agree,
        "final_pred": final_pred,
        "confidence": confidence,
    })

In [11]:
# ── Config ─────────────────────────────────────────────────────────────────
# Point these at your two modality-specific processed datasets.
DATA_PATH_FNIRS = "../cabcsStudy/mne-processing/processed_data/processed_agg_OxySoft_wide_baseline_corrected.csv"
DATA_PATH_EEG = "./data/data_n_back_test/eeg/eeg.parquet"

METADATA_COLS = ["pid", "movement", "block_id", "nback"]
TARGET_COL = "nback"

EEG_DOWNSAMPLE_FACTOR = 10  # Take every Nth row to reduce memory

In [12]:
# ── Train both pipelines independently ───────────────────────────────────────

pipeline_fnirs = train_pipeline(DATA_PATH_FNIRS, "fNIRS")

eeg_df = load_eeg_parquet(DATA_PATH_EEG, EEG_DOWNSAMPLE_FACTOR)
pipeline_eeg = train_pipeline(eeg_df, "EEG")

[fNIRS] Original features: 112
[fNIRS] PCA features: 23
[fNIRS] Explained variance: 0.9530
[fNIRS] Model expects: 23
[EEG parquet] Loaded 143 trials from ./data/data_n_back_test/eeg/eeg.parquet
[EEG parquet] Features per trial: 70
[EEG parquet] Subjects: 16
[EEG parquet] N-back levels: [np.int64(1), np.int64(2), np.int64(3)]
[EEG] Original features: 70
[EEG] PCA features: 3
[EEG] Explained variance: 0.9662
[EEG] Model expects: 3


In [13]:
with open("model_fnirs.pkl", "wb") as f:
    pickle.dump(pipeline_fnirs["model"], f)
with open("scaler_fnirs.pkl", "wb") as f:
    pickle.dump(pipeline_fnirs["scaler"], f)
with open("pca_fnirs.pkl", "wb") as f:
    pickle.dump(pipeline_fnirs["pca"], f)

with open("model_eeg.pkl", "wb") as f:
    pickle.dump(pipeline_eeg["model"], f)
with open("scaler_eeg.pkl", "wb") as f:
    pickle.dump(pipeline_eeg["scaler"], f)
with open("pca_eeg.pkl", "wb") as f:
    pickle.dump(pipeline_eeg["pca"], f)